# 10-714 作业 2

在本作业中，你将在 needle 框架中实现一个神经网络库。提醒：**你必须在云端硬盘中保存副本**。

In [1]:
# 设置作业环境
import sys, os, importlib, needle
import needle.autograd, needle.ops, needle.ops.ops_mathematic, needle.ops.ops_logarithmic
import needle.nn, needle.nn.nn_basic, needle.optim
import needle.init, needle.init.init_basic, needle.init.init_initializers
import needle.data, needle.data.data_basic, needle.data.data_transforms, needle.data.datasets, needle.data.datasets.mnist_dataset
os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())

def reload_needle():
    """修改 needle 源码后，运行此函数重新加载模块，然后可正常调试"""
    importlib.reload(needle.autograd)
    importlib.reload(needle.ops.ops_mathematic)
    importlib.reload(needle.ops.ops_logarithmic)
    importlib.reload(needle.ops)
    importlib.reload(needle.init.init_basic)
    importlib.reload(needle.init.init_initializers)
    importlib.reload(needle.init)
    importlib.reload(needle.nn.nn_basic)
    importlib.reload(needle.nn)
    importlib.reload(needle.optim)
    importlib.reload(needle.data.datasets.mnist_dataset)
    importlib.reload(needle.data.datasets)
    importlib.reload(needle.data.data_transforms)
    importlib.reload(needle.data.data_basic)
    importlib.reload(needle.data)
    importlib.reload(needle)
    # 重载 apps 目录的模块
    for mod_name in list(sys.modules.keys()):
        if 'mlp_resnet' in mod_name:
            importlib.reload(sys.modules[mod_name])
    # 清除已导入的测试模块缓存，确保下次 pytest 使用新的 needle 类
    for mod_name in list(sys.modules.keys()):
        if 'test_nn' in mod_name or 'test_optim' in mod_name or 'test_mlp' in mod_name:
            del sys.modules[mod_name]
    print('needle 已重载，可以调试')


In [7]:
# 修改 needle 源码后，运行此单元格重新加载（可调试）
reload_needle()

needle 已重载，可以调试


### 设置一些变量

In [49]:
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW2_NAME = "hw2"

## 问题 0

本作业建立在作业 1 的基础上。

**首先，在你的作业 2 目录中，从作业 1 复制文件 `python/needle/autograd.py`、`python/needle/ops/ops_mathematic.py`。**

***注意***：Tensor 的默认数据类型是 `float32`。如果你想更改数据类型，可以通过在 `Tensor` 构造函数中设置 `dtype` 参数来实现。例如，`Tensor([1, 2, 3], dtype='float64')` 将创建一个 `float64` 数据类型的张量。
在本作业中，**确保你创建的任何张量都使用 `float32` 数据类型，以避免自动评分器出现问题**。

In [50]:
sys.path.append('./python')
sys.path.append('./apps')

## 问题 1

在第一个问题中，你将实现几种不同的权重初始化方法。这将在 `python/needle/init/init_initializers.py` 文件中完成，该文件包含许多使用各种随机和常量初始化来初始化 needle Tensor 的例程。按照现有初始化器的相同方法（你将需要调用 `python/needle/init/init_basic.py` 中实现的 `init.rand` 或 `init.randn`），实现以下常见的初始化方法。在所有情况下，函数应返回 `fan_in` × `fan_out` 的 2D 张量（可以通过 reshape 扩展到其他大小）。


### Xavier 均匀分布
`xavier_uniform(fan_in, fan_out, gain=1.0, **kwargs)`

按照[Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf)中描述的方法，使用均匀分布填充输入 Tensor。结果 Tensor 的值将从 $\mathcal{U}(-a, a)$ 中采样，其中

$$a = \text{gain} \times \sqrt{\frac{6}{\text{fan\_in} + \text{fan\_out}}}$$

将剩余的 `**kwargs` 参数传递给相应的 `init` 随机调用。

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `gain` - 可选的缩放因子
___

### Xavier 正态分布
`xavier_normal(fan_in, fan_out, gain=1.0, **kwargs)`

按照[Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf)中描述的方法，使用正态分布填充输入 Tensor。结果 Tensor 的值将从 $\mathcal{N}(0, \text{std}^2)$ 中采样，其中

$$\mathrm{std} = \mathrm{gain} \times \sqrt{\frac{2}{\mathrm{fan}_{in} + \mathrm{fan}_{out}}}$$

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `gain` - 可选的缩放因子
___

### Kaiming 均匀分布
`kaiming_uniform(fan_in, fan_out, nonlinearity="relu", **kwargs)`

按照[Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification](https://arxiv.org/pdf/1502.01852.pdf)中描述的方法，使用均匀分布填充输入 Tensor。结果 Tensor 的值将从 $\mathcal{U}(-\text{bound}, \text{bound})$ 中采样，其中

$$\mathrm{bound} = \mathrm{gain} \times \sqrt{\frac{3}{\mathrm{fan}_{in}}}$$

使用 ReLU 的推荐 gain 值：$\text{gain}=\sqrt{2}$。

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `nonlinearity` - 非线性函数
___

### Kaiming 正态分布
`kaiming_normal(fan_in, fan_out, nonlinearity="relu", **kwargs)`

按照[Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification](https://arxiv.org/pdf/1502.01852.pdf)中描述的方法，使用正态分布填充输入 Tensor。结果 Tensor 的值将从 $\mathcal{N}(0, \text{std}^2)$ 中采样，其中

$$\mathrm{std} = \frac{\mathrm{gain}}{\sqrt{\mathrm{fan}_{in}}}$$

使用 ReLU 的推荐 gain 值：$\text{gain}=\sqrt{2}$。

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `nonlinearity` - 非线性函数

In [51]:
import pytest
pytest.main(['-v', '-k', 'test_init'])

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 89 deselected / 4 selected

tests/hw2/test_nn_and_optim.py::test_init_kaiming_uniform PASSED         [ 25%]
tests/hw2/test_nn_and_optim.py::test_init_kaiming_normal PASSED          [ 50%]
tests/hw2/test_nn_and_optim.py::test_init_xavier_uniform PASSED          [ 75%]
tests/hw2/test_nn_and_optim.py::test_init_xavier_normal PASSED           [100%]

======================= 4 passed, 89 deselected in 3.77s =======================


<ExitCode.OK: 0>

In [52]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "init"

## 问题 2

在此问题中，你将在 `python/needle/nn/nn_basic.py` 中实现额外的模块。具体来说，对于下面描述的以下模块，在构造函数中初始化模块的任何变量，并填写 `forward` 方法。**注意：** 确保你使用刚刚实现的 `init` 函数来初始化参数，并且不要忘记传递 `dtype` 参数。
___

### Linear
`needle.nn.Linear(in_features, out_features, bias=True, device=None, dtype="float32")`

对传入数据应用线性变换：$y = xA^T + b$。输入形状为 $(N, H_{\text{in}})$，其中 $H_{\text{in}}=\text{in\_features}$。输出形状为 $(N, H_{\text{out}})$，其中 $H_{\text{out}}=\text{out\_features}$。

**注意要显式地将偏置项广播到正确的形状——Needle 不支持隐式广播。**

**注意：对于包括此层在内的所有层，你应在偏置 Tensor 之前初始化权重 Tensor，并且应仅使用 `init` 中的函数初始化所有 Parameter**。此初始化顺序要求存在是因为 mugrade 上的测试用例答案是假设权重在偏置参数之前初始化的。虽然在权重之前初始化偏置在算法上仍然是正确的，但模型解决方案和预期测试输出是使用权重优先约定生成的。因此，要通过 mugrade 上的自动测试，你必须遵循此特定初始化顺序。如果你对本作业其他部分中应首先初始化哪个层或参数有任何歧义，请在 Ed 上提出请求以获取澄清。

##### 参数
- `in_features` - 每个输入样本的大小
- `out_features` - 每个输出样本的大小
- `bias` - 如果设为 `False`，该层将不会学习加性偏置。

##### 变量
- `weight` - 形状为 (`in_features`, `out_features`) 的可学习权重。应使用 `fan_in = in_features` 的 **Kaiming 均匀分布初始化**
- `bias` - 形状为 (`out_features`) 的可学习偏置。应使用 `fan_in = out_features` 的 Kaiming 均匀分布初始化。**注意 fan_in 选择的差异，由于它们的相对大小**。


**注意：** 确保将所有必要变量（如 `weight`、`bias`）封装在 **`Parameter` 类**中，以便接下来实现的优化器可以看到它们。**你可以阅读 `python/needle/nn/nn_basic.py` 中的 `class Parameter` 和函数 `unpack_params` 以了解更多。**

In [53]:
import pytest
pytest.main(['-v', '-k', 'test_nn_linear'])

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 85 deselected / 8 selected

tests/hw2/test_nn_and_optim.py::test_nn_linear_weight_init_1 PASSED      [ 12%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_bias_init_1 PASSED        [ 25%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_forward_1 PASSED          [ 37%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_forward_2 PASSED          [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_forward_3 PASSED          [ 62%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_backward_1 PASSED         [ 75%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_backward_2 PASSED         [ 87%]
tests/hw2/test_nn_and_optim.py::test_nn_linear_backward_3 PASSED         [100%]

======================= 8 passed, 85 deselecte

<ExitCode.OK: 0>

In [54]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_linear"

### ReLU
`needle.nn.ReLU()`

逐元素应用修正线性单元函数：
$ReLU(x) = max(0, x)$。

如果你之前用 ReLU 自身实现了 ReLU 的反向传播，请注意这在数值上不稳定，可能会在后面导致问题。
相反，我们可以将 ReLU 的导数写为 $I\{x>0\}$，其中，在本作业中，我们任意决定并固定约定：在 $x=0$ 处的导数为 0。
（这是一个_次可微_函数。）

___

In [55]:
import pytest
pytest.main(['-v', '-k', 'test_nn_relu'])

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 91 deselected / 2 selected

tests/hw2/test_nn_and_optim.py::test_nn_relu_forward_1 PASSED            [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_relu_backward_1 PASSED           [100%]

======================= 2 passed, 91 deselected in 0.04s =======================


<ExitCode.OK: 0>

In [56]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_relu"

### Sequential
`needle.nn.Sequential(*modules)`

按照传递给构造函数的顺序将一系列模块应用于输入，并返回最后一个模块的输出。
这些应保存在 `.module` 属性中：你_不应_重新定义任何魔术方法如 `__getitem__`，因为这可能与我们的测试不兼容。

##### 参数
- `*modules` - 任意数量的 `needle.nn.Module` 类型的模块

___

In [57]:
import pytest
pytest.main(['-v', '-k', 'test_nn_sequential'])

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 91 deselected / 2 selected

tests/hw2/test_nn_and_optim.py::test_nn_sequential_forward_1 PASSED      [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_sequential_backward_1 PASSED     [100%]

======================= 2 passed, 91 deselected in 0.03s =======================


<ExitCode.OK: 0>

In [58]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_sequential"

### LogSumExp

`needle.ops.LogSumExp(axes)`

通过减去最大元素，对输入应用数值稳定的 log-sum-exp 函数。你将需要在 `python/needle/ops/ops_logarithmic.py` 文件中实现此操作和下一个操作。

\begin{equation}
\text{LogSumExp}(z) = \log (\sum_{i} \exp (z_i - \max{z})) + \max{z}
\end{equation}

#### 参数
- `axes` - 要求和并取最大元素的轴元组。使用与 `needle.ops.Summation()` 相同的约定

##### 为什么用这种公式？处理数值稳定性

- **朴素定义：**
  定义 LogSumExp 最直接的方式是

  $$
  \log \left(\sum_i \exp(z_i)\right)
  $$

- **问题：**
  这种朴素计算容易出现数值不稳定。
  - 如果某个 $z_i$ 非常大（例如 $1000$），则 $\exp(z_i)$ 会溢出为 $\infty$。
  - 如果某个 $z_i$ 非常小（例如 $-1000$），则 $\exp(z_i)$ 会下溢为 $0$。
  两种情况都会在浮点运算中扭曲结果。

- **修复方法：**
  为了避免这种情况，我们提取最大元素 $M = \max(z)$：

  $$
  \log \left(\sum_i \exp(z_i)\right)
  = \log \left(\exp(M)\sum_i \exp(z_i - M)\right)
  = M + \log \left(\sum_i \exp(z_i - M)\right).
  $$

  现在所有指数最多为 $\exp(0) = 1$，因此完全避免了溢出。

- **那下溢呢？**
  如果 $z_i - M$ 非常负（例如 $-1000$），下溢仍可能发生，因为在浮点中 $\exp(-1000) \approx 0$。
  然而，这不是问题：与最大值相比，这些项已经可以忽略不计，不会对总和产生有意义的影响。


以下博客文章也是很好的参考：https://indii.org/blog/gradients-of-softmax-and-logsumexp/
___

In [2]:
import pytest
pytest.main(['-v', '-k', 'test_op_logsumexp'])

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 83 deselected / 10 selected

tests/hw2/test_nn_and_optim.py::test_op_logsumexp_forward_1 PASSED       [ 10%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_forward_2 PASSED       [ 20%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_forward_3 PASSED       [ 30%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_forward_4 PASSED       [ 40%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_forward_5 PASSED       [ 50%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_backward_1 PASSED      [ 60%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_backward_2 PASSED      [ 70%]
tests/hw2/test_nn_and_optim.py::test_op_logsumexp_backward_3 PASSED      [ 80%]
tests/hw2/test_nn_and_optim.py::test_op_logsum

<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "op_logsumexp"

### LogSoftmax

`needle.ops.LogSoftmax(axes)`

通过减去最大元素，对输入应用数值稳定的 logsoftmax 函数。
对于此问题，你可以假设输入 NDArray 是 2 维的，我们在 `axis=1` 上做 softmax。

\begin{equation}
\text{LogSoftmax}(z) = \log \left(\frac{\exp(z_i - \max z)}{\sum_{i}\exp(z_i - \max z)}\right) = z - \text{LogSumExp}(z)
\end{equation}
___

needle 已重载，可以调试


In [21]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_op_logsoftmax'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... 

collected 93 items / 90 deselected / 3 selected

tests/hw2/test_nn_and_optim.py::test_op_logsoftmax_forward_1 PASSED      [ 33%]
tests/hw2/test_nn_and_optim.py::test_op_logsoftmax_stable_forward_1 PASSED [ 66%]
tests/hw2/test_nn_and_optim.py::test_op_logsoftmax_backward_1 PASSED     [100%]

======================= 3 passed, 90 deselected in 0.02s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "op_logsoftmax"

### SoftmaxLoss

`needle.nn.SoftmaxLoss()`，在 `python/needle/nn/nn_basic.py` 中

应用如下定义的 softmax 损失（与作业 1 中实现的相同），输入为 logit 的 Tensor 和真实标签的 Tensor（以数字列表表示，*不是*独热编码）。

请注意，你现在可以使用 `init.one_hot` 函数，而不是自己编写。注意：你将需要使用刚刚实现的数值稳定的 logsumexp 运算符来完成此目的。

\begin{equation}
\ell_\text{softmax}(z,y) = \log \sum_{i=1}^k \exp z_i - z_y
\end{equation}

___

In [33]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_nn_softmax_loss'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 89 deselected / 4 selected

tests/hw2/test_nn_and_optim.py::test_nn_softmax_loss_forward_1 PASSED    [ 25%]
tests/hw2/test_nn_and_optim.py::test_nn_softmax_loss_forward_2 PASSED    [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_softmax_loss_backward_1 PASSED   [ 75%]
tests/hw2/test_nn_and_optim.py::test_nn_softmax_loss_backward_2 PASSED   [100%]

======================= 4 passed, 89 deselected in 0.02s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_softmax_loss"

### LayerNorm1d
`needle.nn.LayerNorm1d(dim, eps=1e-5, device=None, dtype="float32")`

按照论文 [Layer Normalization](https://arxiv.org/abs/1607.06450) 中描述的方法，对小批量输入应用层归一化。

\begin{equation}
y = w \circ \frac{x_i - \textbf{E}[x]}{((\textbf{Var}[x]+\epsilon)^{1/2})} + b
\end{equation}

其中 $\textbf{E}[x]$ 表示输入的经验均值，$\textbf{Var}[x]$ 表示其经验方差（注意这里我们使用方差的'有偏'估计，即除以 $N$ 而非 $N-1$），$w$ 和 $b$ 分别表示可学习的标量权重和偏置。注意你可以假设此层的输入是 2D 张量，第一维是批次，第二维是特征。你可能需要在应用之前广播权重和偏置。

##### 参数
- `dim` - 通道数
- `eps` - 为数值稳定性添加到分母的值。

##### 变量
- `weight` - 大小为 `dim` 的可学习权重，元素初始化为 1。
- `bias` - 形状为 `dim` 的可学习偏置，元素初始化为 0。
___

In [12]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_nn_layernorm'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 86 deselected / 7 selected

tests/hw2/test_nn_and_optim.py::test_nn_layernorm_forward_1 PASSED       [ 14%]
tests/hw2/test_nn_and_optim.py::test_nn_layernorm_forward_2 PASSED       [ 28%]
tests/hw2/test_nn_and_optim.py::test_nn_layernorm_forward_3 PASSED       [ 42%]
tests/hw2/test_nn_and_optim.py::test_nn_layernorm_backward_1 PASSED      [ 57%]
tests/hw2/test_nn_and_optim.py::test_nn_layernorm_backward_2 PASSED      [ 71%]
tests/hw2/test_nn_and_optim.py::test_nn_layernorm_backward_3 PASSED      [ 85%]
tests/hw2/test_nn_and_optim.py::test_nn_layernorm_backward_4 PASSED      [100%]

======================= 7 passed, 86 deselected in 0.05s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_layernorm"

### Flatten
`needle.nn.Flatten()`

接收形状为 `(B,X_0,X_1,...)` 的张量，展平所有非批次维度，使输出形状为 `(B, X_0 * X_1 * ...)`

In [ ]:
import pytest
pytest.main(['-v', '-k', 'test_nn_flatten'])

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_flatten"

### BatchNorm1d
`needle.nn.BatchNorm1d(dim, eps=1e-5, momentum=0.1, device=None, dtype="float32")`

按照论文 [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167) 中描述的方法，对小批量输入应用批次归一化。

\begin{equation}
y = w \circ \frac{z_i - \textbf{E}[x]}{((\textbf{Var}[x]+\epsilon)^{1/2})} + b
\end{equation}

但这里的均值和方差指的是_批次维度_上的均值和方差。该函数还为每层的所有特征计算均值/方差的运行平均值 $\hat{\mu}, \hat{\sigma}^2$，并在测试时使用这些量进行归一化：

\begin{equation}
y = \frac{(x - \hat{mu})}{((\hat{\sigma}^2_{i+1})_j+\epsilon)^{1/2}}
\end{equation}


BatchNorm 在测试时使用均值和方差的运行估计而非批次统计，即在 BatchNorm 层的 `training` 标志被调用 `model.eval()` 后为 false。

要计算运行估计，你可以使用方程 $$\hat{x_{new}} = (1 - m) \hat{x_{old}} + mx_{observed}$$
其中 $m$ 是动量。

##### 参数
- `dim` - 输入维度
- `eps` - 为数值稳定性添加到分母的值。
- `momentum` - 用于运行均值和运行方差计算的值。

##### 变量
- `weight` - 大小为 `dim` 的可学习权重，元素初始化为 1。
- `bias` - 大小为 `dim` 的可学习偏置，元素初始化为 0。
- `running_mean` - 评估时使用的运行均值，元素初始化为 0。
- `running_var` - 评估时使用的运行（无偏）方差，元素初始化为 1。

___

In [2]:
import pytest
pytest.main(['-v', '-k', 'test_nn_batchnorm'])

============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 85 deselected / 8 selected

tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_check_model_eval_switches_training_flag_1 PASSED [ 12%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_forward_1 PASSED       [ 25%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_forward_affine_1 PASSED [ 37%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_backward_1 PASSED      [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_backward_affine_1 PASSED [ 62%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_running_mean_1 PASSED  [ 75%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_running_var_1 PASSED   [ 87%]
tests/hw2/test_nn_and_optim.py::test_nn_batchnorm_running_grad_1 PASSED  [100%]

=================

<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_batchnorm"

### Dropout
`needle.nn.Dropout(p = 0.5)`

在训练期间，使用伯努利分布采样，以概率 `p` 随机将输入张量的某些元素置零。这已被证明是一种有效的正则化技术，可防止神经元的共适应，如论文 [Improving neural networks by preventing co-adaptation of feature detectors](https://arxiv.org/abs/1207.0580) 中所述。在评估期间，该模块只计算恒等函数。

\begin{equation}
\hat{z}_{i+1} = \sigma_i (W_i^T z_i + b_i) \\
(z_{i+1})_j =
    \begin{cases}
    (\hat{z}_{i+1})_j /(1-p) & \text{概率 } 1-p \\
    0 & \text{概率 } p \\
    \end{cases}
\end{equation}

除以 \(1-p\) 保持期望激活不变，因为

$$
\mathbb{E}\big[\operatorname{Dropout}((\hat{z}_{i+1})_j)\big]
= (1-p)\,\frac{(\hat{z}_{i+1})_j}{1-p} + p \cdot 0
= (\hat{z}_{i+1})_j
$$

**重要**：如果 Dropout 模块的标志 `training=False`，你不应该'丢弃'任何激活。也就是说，dropout 仅在训练期间适用，不在评估期间。注意 `training` 是 `nn.Module` 中的标志。

##### 参数
- `p` - 元素被置零的概率。

`python/needle/init/init_basic.py` 中的工具在实现 Dropout 时可能会有帮助。
___

In [7]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_nn_dropout'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 91 deselected / 2 selected

tests/hw2/test_nn_and_optim.py::test_nn_dropout_forward_1 PASSED         [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_dropout_backward_1 PASSED        [100%]

======================= 2 passed, 91 deselected in 0.02s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_dropout"

### Residual
`needle.nn.Residual(fn: Module)`

给定模块 $\mathcal{F}$ 和输入 Tensor $x$，应用残差或跳跃连接，返回 $\mathcal{F}(x) + x$。
##### 参数
- `fn` - `needle.nn.Module` 类型的模块

In [9]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_nn_residual'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 91 deselected / 2 selected

tests/hw2/test_nn_and_optim.py::test_nn_residual_forward_1 PASSED        [ 50%]
tests/hw2/test_nn_and_optim.py::test_nn_residual_backward_1 PASSED       [100%]

======================= 2 passed, 91 deselected in 0.03s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_residual"

## 问题 3

在 `python/needle/optim.py` 中实现以下优化器的 `step` 函数。
确保你的优化器_不会_原地修改张量的梯度。

我们包含了一些测试来确保你没有消耗过多的内存，如果你没有在正确的地方使用 `.data` 或 `.detach()`，可能会发生这种情况，从而构建越来越大的计算图（不仅在优化器中，在前面的模块中也是如此）。
你可以自行决定忽略这些包含 `memory_check` 字符串的测试。
___

### SGD
`needle.optim.SGD(params, lr=0.01, momentum=0.0, weight_decay=0.0)`

实现随机梯度下降（可选带动量，如下所示为 $\beta$）。

\begin{equation}
\begin{split}
    u_{t+1} &= \beta u_t + (1-\beta) \nabla_\theta f(\theta_t) \\
    \theta_{t+1} &= \theta_t - \alpha u_{t+1}
\end{split}
\end{equation}

##### 参数
- `params` - 要优化的 `needle.nn.Parameter` 类型的参数可迭代对象
- `lr` (*float*) - 学习率
- `momentum` (*float*) - 动量因子
- `weight_decay` (*float*) - 权重衰减（L2 惩罚）

本作业可以跳过 `clip_grad_norm` 的实现。
___

In [16]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_optim_sgd'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... 

collected 93 items / 87 deselected / 6 selected

tests/hw2/test_nn_and_optim.py::test_optim_sgd_vanilla_1 PASSED          [ 16%]
tests/hw2/test_nn_and_optim.py::test_optim_sgd_momentum_1 PASSED         [ 33%]
tests/hw2/test_nn_and_optim.py::test_optim_sgd_weight_decay_1 PASSED     [ 50%]
tests/hw2/test_nn_and_optim.py::test_optim_sgd_momentum_weight_decay_1 PASSED [ 66%]
tests/hw2/test_nn_and_optim.py::test_optim_sgd_layernorm_residual_1 PASSED [ 83%]
tests/hw2/test_nn_and_optim.py::test_optim_sgd_z_memory_check_1 PASSED   [100%]

======================= 6 passed, 87 deselected in 0.16s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "optim_sgd"

### Adam
`needle.optim.Adam(params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0)`

实现 Adam 算法，由论文 [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980) 提出。

\begin{equation}
\begin{split}
u_{t+1} &= \beta_1 u_t + (1-\beta_1) \nabla_\theta f(\theta_t) \\
v_{t+1} &= \beta_2 v_t + (1-\beta_2) (\nabla_\theta f(\theta_t))^2 \\
\hat{u}_{t+1} &= u_{t+1} / (1 - \beta_1^t) \quad \text{(偏差修正)} \\
\hat{v}_{t+1} &= v_{t+1} / (1 - \beta_2^t) \quad \text{(偏差修正)}\\
\theta_{t+1} &= \theta_t - \alpha \hat{u_{t+1}}/(\hat{v}_{t+1}^{1/2}+\epsilon)
\end{split}
    \end{equation}

**重要：** 注意你是否在应用偏差修正。

##### 参数
- `params` - 要优化的 `needle.nn.Parameter` 类型的参数可迭代对象
- `lr` (*float*) - 学习率
- `beta1` (*float*) - 用于计算梯度运行平均值的系数
- `beta2` (*float*) - 用于计算梯度平方运行平均值的系数
- `eps` (*float*) - 添加到分母以提高数值稳定性的项
- `weight_decay` (*float*) - 权重衰减（L2 惩罚）

**提示**：为帮助处理内存问题，尝试理解如何使用 `.data` 或 `.detach()`

In [33]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_optim_adam'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 86 deselected / 7 selected

tests/hw2/test_nn_and_optim.py::test_optim_adam_1 PASSED                 [ 14%]
tests/hw2/test_nn_and_optim.py::test_optim_adam_weight_decay_1 PASSED    [ 28%]
tests/hw2/test_nn_and_optim.py::test_optim_adam_batchnorm_1 PASSED       [ 42%]
tests/hw2/test_nn_and_optim.py::test_optim_adam_batchnorm_eval_mode_1 PASSED [ 57%]
tests/hw2/test_nn_and_optim.py::test_optim_adam_layernorm_1 PASSED       [ 71%]
tests/hw2/test_nn_and_optim.py::test_optim_adam_weight_decay_bias_correction_1 PASSED [ 85%]
tests/hw2/test_nn_and_optim.py::test_optim_adam_z_memory_check_1 PASSED  [100%]

======================= 7 passed, 86 deselected in 0.16s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "optim_adam"

## 问题 4

在此问题中，你将实现两个数据原语：`needle.data.DataLoader` 和 `needle.data.Dataset`。`Dataset` 存储样本及其对应标签，`DataLoader` 在 `Dataset` 周围包装一个可迭代对象，以便于访问样本。

对于此问题，你将在 `python/needle/data` 目录中工作。

### 变换

首先，我们将实现一些在处理图像时有用的变换。目前我们将坚持使用水平翻转和随机裁剪。在 `needle/data/data_transforms.py` 中填写以下函数。
___

#### RandomFlipHorizontal
`needle.data.RandomFlipHorizontal(p = 0.5)`

以概率 `p` 水平翻转图像。

##### 参数
- `p` (*float*) - 翻转输入图像的概率。
___

#### RandomCrop
`needle.data.RandomCrop(padding=3)`

在图像的所有边添加填充，然后在随机位置将图像裁剪回原始大小。返回与原始图像大小相同的图像。

##### 参数
- `padding` (*int*) - 图像每个边的填充。

In [42]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'flip_horizontal'])
pytest.main(['-v', '-k', 'random_crop'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 92 deselected / 1 selected

tests/hw2/test_data.py::test_flip_horizontal PASSED                      [100%]

======================= 1 passed, 92 deselected in 0.03s =======================
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 92 deselected / 1 selected

tests/hw2/test_data.py::test_random_crop PASSED                          [100%]

======================= 1 passed, 92 deselected in 0.02s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "flip_horizontal"
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "random_crop"

Dataset 负责你的数据是什么（例如图像/标签对）以及如何获取单个样本。Dataloader 负责如何将数据馈送到训练中（例如批处理、打乱、遍历 epoch）。

### Dataset

`Dataset` 类的每个**子类**必须实现三个函数：`__init__`、`__len__` 和 `__getitem__`。
* `__init__` 函数初始化图像、标签和变换。
* `__len__` 函数返回数据集中的样本数。
* `__getitem__` 函数从数据集中检索给定索引 `idx` 的样本，**对图像调用变换函数（如适用）**，将图像和标签转换为 numpy 数组（数据将在其他地方转换为 Tensor）。`__getitem__` 和 `__next__` 的输出应为 NDArray，你应遵循形状规则，使得你访问的是大小为 (数据点数, 特征维度 1, 特征维度 2, ...) 的数组。

在 `needle/data/datasets/mnist_dataset.py` 的 `MNISTDataset` 类中填写这些函数。你可以使用上一个作业中的 `parse_mnist` 解决方案作为 `__init__` 函数。

### MNISTDataset（Dataset 的子类）
`needle.data.MNISTDataset(image_filename, label_filename, transforms)`

##### 参数
- `image_filename` - 包含图像的文件路径
- `label_filename` - 包含标签的文件路径
- `transforms` - 应用于数据的可选变换列表

In [51]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_mnist_dataset'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 92 deselected / 1 selected

tests/hw2/test_data.py::test_mnist_dataset PASSED                        [100%]

======================= 1 passed, 92 deselected in 0.98s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "mnist_dataset"

### Dataloader

`needle.data.Dataloader(dataset: Dataset, batch_size: Optional[int] = 1, shuffle: bool = False)`


在 `needle/data/data_basic.py` 中，Dataloader 类提供了一个接口，用于组装适合使用基于 SGD 方法训练的小批量示例，由 `Dataset` 对象支持。为了构建典型的 Dataloader 接口（允许用户遍历数据集中的所有小批量），你需要在类中实现 `__iter__()` 和 `__next__()` 调用：
* `__iter__()` 在新 epoch 开始时调用（即每当你开始遍历 dataloader 时）。
* `__next__()` 每个批次调用一次，直到 epoch 结束。

请注意，后续调用 next 将要求你返回后续批次，因此 next 不是纯函数。

##### 用途

组合数据集和采样器，并提供对给定数据集的可迭代对象。

##### 参数
- `dataset` - `needle.data.Dataset` - 数据集
- `batch_size` - `int` - 提供数据的批次大小
- `shuffle` - `bool` - 设为 `True` 以在**每个 epoch 开始时**重新打乱数据，默认 `False`。
___


In [76]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_dataloader'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 91 deselected / 2 selected

tests/hw2/test_data.py::test_dataloader_mnist PASSED                     [ 50%]
tests/hw2/test_data.py::test_dataloader_ndarray PASSED                   [100%]

======================= 2 passed, 91 deselected in 5.94s =======================


<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "dataloader"

## 问题 5

既然你已经实现了神经网络库的所有必要组件，让我们构建并训练一个 MLP ResNet。对于此问题，你将在 `apps/mlp_resnet.py` 中工作。首先，按照下面描述填写 `ResidualBlock` 和 `MLPResNet` 函数：

### ResidualBlock
`ResidualBlock(dim, hidden_dim, norm=nn.BatchNorm1d, drop_prob=0.1)`

实现如下所示的残差块：

<p align="center">
  <img src="https://github.com/dlsyscourse/hw2/blob/f4c994506f2c76d7fdcc5a711a483e31b189afaa/figures/residualblock.png?raw=true" alt="残差块"/>
</p>

**注意**：如果图片无法渲染，请查看 `figures` 目录中的图片。

其中第一个线性层的 `in_features=dim`，`out_features=hidden_dim`，最后一个线性层的 `out_features=dim`。返回类型为 `nn.Module` 的块。

##### 参数
- `dim` (*int*) - 输入维度
- `hidden_dim` (*int*) - 隐藏维度
- `norm` (*nn.Module*) - 归一化方法
- `drop_prob` (*float*) - dropout 概率

___

### MLPResNet
`MLPResNet(dim, hidden_dim=100, num_blocks=3, num_classes=10, norm=nn.BatchNorm1d, drop_prob=0.1)`

实现如下所示的 MLP ResNet：

<p align="center">
  <img src="https://github.com/dlsyscourse/hw2/blob/f4c994506f2c76d7fdcc5a711a483e31b189afaa/figures/mlp_resnet.png?raw=true" alt="MLP Resnet"/>
</p>

其中第一个线性层的 `in_features=dim`，`out_features=hidden_dim`，每个 ResidualBlock 的 `dim=hidden_dim`，`hidden_dim=hidden_dim//2`。返回类型为 `nn.Module` 的网络。

##### 参数
- `dim` (*int*) - 输入维度
- `hidden_dim` (*int*) - 隐藏维度
- `num_blocks` (*int*) - ResidualBlock 的数量
- `num_classes` (*int*) - 类别数
- `norm` (*nn.Module*) - 归一化方法
- `drop_prob` (*float*) - dropout 概率 (0.1)

**注意**：模块的初始化顺序应与 Resnet 中的执行顺序匹配。
___

一旦你有了正确的深度学习模型架构，让我们使用新的神经网络库组件来训练网络。具体来说，实现函数 `epoch` 和 `train_mnist`。

### Epoch

`epoch(dataloader, model, opt=None)`

执行一个 epoch 的训练或评估，遍历整个训练数据集一次（就像之前作业中的 `nn_epoch` 一样）。返回平均错误率（*float*）和所有样本的平均损失（*float*）。如果给定了 `opt`，则在函数开始时将模型设为 `training` 模式；如果 `opt` 为 `None`，则将模型设为 `eval`。设置模式时，使用 `.train()` 和 `.eval()` 而不是修改 training 属性。

##### 参数
- `dataloader` (*`needle.data.DataLoader`*) - 从训练数据集返回样本的 dataloader
- `model` (*`needle.nn.Module`*) - 神经网络
- `opt` (*`needle.optim.Optimizer`*) - 优化器实例，或 `None`

___

### 训练 MNIST

`train_mnist(batch_size=100, epochs=10, optimizer=ndl.optim.Adam, lr=0.001, weight_decay=0.001, hidden_dim=100, data_dir="data")`

初始化一个训练 dataloader（`shuffle` 设为 `True`）和一个 MNIST 数据的测试 dataloader，使用给定的优化器（如果 `opt` 不为 None）和 softmax 损失训练 `MLPResNet` 给定的 epoch 数。返回最后一个 epoch 中计算的训练错误率、训练损失、测试错误率、测试损失的元组。如果未指定任何参数，则使用默认参数。

##### 参数
- `batch_size` (*int*) - 训练和测试 dataloader 使用的批次大小
- `epochs` (*int*) - 训练的 epoch 数
- `optimizer` (*`needle.optim.Optimizer` 类型*) - 要使用的优化器类型
- `lr` (*float*) - 学习率
- `weight_decay` (*float*) - 权重衰减
- `hidden_dim` (*int*) - `MLPResNet` 的隐藏维度
- `data_dir` (*str*) - 包含 MNIST 图像/标签文件的目录

In [6]:
reload_needle()
import pytest
pytest.main(['-v', '-k', 'test_mlp'])

needle 已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw2
collecting ... collected 93 items / 83 deselected / 10 selected

tests/hw2/test_nn_and_optim.py::test_mlp_residual_block_num_params_1 PASSED [ 10%]
tests/hw2/test_nn_and_optim.py::test_mlp_residual_block_num_params_2 PASSED [ 20%]
tests/hw2/test_nn_and_optim.py::test_mlp_residual_block_forward_1 PASSED [ 30%]
tests/hw2/test_nn_and_optim.py::test_mlp_resnet_num_params_1 PASSED      [ 40%]
tests/hw2/test_nn_and_optim.py::test_mlp_resnet_num_params_2 PASSED      [ 50%]
tests/hw2/test_nn_and_optim.py::test_mlp_resnet_forward_1 PASSED         [ 60%]
tests/hw2/test_nn_and_optim.py::test_mlp_resnet_forward_2 PASSED         [ 70%]
tests/hw2/test_nn_and_optim.py::test_mlp_train_epoch_1 PASSED            [ 80%]
tests/hw2/test_nn_and_op

<ExitCode.OK: 0>

In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "mlp_resnet"

我们鼓励你尝试 `mlp_resnet.py` 训练脚本。
你可以研究在 Linear 层上使用不同初始化器的效果，
增加 dropout 概率，
或添加变换（通过 Dataset 的 `transforms=` 关键字参数传递列表）
如随机裁剪。